# Knowledge distillation into a 17M encoder with 10% of SST-2 labels

Teacher: [Ettin-encoder-400m](https://huggingface.co/jhu-clsp/ettin-encoder-400m). Student: [Ettin-encoder-17m](https://huggingface.co/jhu-clsp/ettin-encoder-17m). Both are ModernBERT-architecture encoders from one pretraining recipe ([Weller et al., 2025](https://arxiv.org/abs/2507.11412)) with one tokenizer. Task: GLUE SST-2, binary sentiment; the GLUE validation split (872 sentences) is the test set. Code and plan: [project/](https://github.com/PE51K/itmo-ai-talent-hub-efficient-models-course/tree/main/project).

The notebook runs every run of `run_list()` in `kd.py`, skips runs whose result is already in `results/runs/`, and stops starting new runs after `MAX_HOURS`.

| Group | Runs | What it measures | T4 time, estimate |
|---|---|---|---|
| 10% table | `teacher_p10`, `hard_p10`, `hard_p100`, `kd_p10`, `kdu_p10` | KD gain at a 10% label budget | ~1 h |
| Full labels | `teacher_p100`, `kd_p100` | KD gain with all labels | ~45 min |
| Label curve | teacher, `hard`, `kdu` at 1%, 5%, 25% | gain vs label budget | ~1 h |
| Selection | `teacher_kc10`, `hard_kc10`, `kdu_kc10` | k-center vs random 10% | ~25 min |
| Benchmark | `bench` | parameters, CPU latency, T4 throughput | ~2 min |

Student runs have 3 seeds (`_s0`, `_s1`, `_s2`), teacher runs 1. The times are estimates from typical T4 fp16 throughput for 400M and 17M encoders, not measurements; `kd.py` prints the time of every run.

## Settings and resuming

Kaggle: Accelerator GPU T4 x2 (one GPU is used) or P100, Internet on. Click Save Version -> Save & Run All; a saved version runs in the background for up to 12 hours, and its `results/` and `results.zip` are in the Output tab.

To continue after a stopped or failed version, add the previous version's output as input (Add Input -> Your Work -> this notebook) and run again: the resume cell copies every `results/` folder and `results.zip` it finds under `/kaggle/input` into the working directory, and finished runs are skipped.

Colab: Runtime -> Change runtime type -> T4 GPU. To resume, upload `results.zip` to `/content` before running. Download `/content/results.zip` at the end; a Colab session loses its files when it disconnects.

In [ ]:
MAX_HOURS = 11.0 # no run starts after this, Kaggle stops a session at 12 hours
RUNS = "" # space-separated run names from kd.py, empty runs everything

In [ ]:
import os
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
!git clone --depth 1 https://github.com/PE51K/itmo-ai-talent-hub-efficient-models-course.git /tmp/repo || git -C /tmp/repo pull
%cd /tmp/repo/project
!pip install -q "transformers>=4.48" datasets
# kd.py writes to project/results, which points to the output directory
(WORK / "results").mkdir(exist_ok=True)
!ln -sfn {WORK}/results results

In [ ]:
import shutil
import zipfile

for src in Path("/kaggle/input").glob("**/results") if Path("/kaggle/input").exists() else []:
    if (src / "runs").is_dir():
        shutil.copytree(src, WORK / "results", dirs_exist_ok=True)
        print("copied", src)
zips = list(Path("/kaggle/input").glob("**/results.zip")) if Path("/kaggle/input").exists() else []
zips += [p for p in [Path("/content/results.zip")] if p.exists()]
for z in zips:
    zipfile.ZipFile(z).extractall(WORK)
    print("unpacked", z)
print(len(list((WORK / "results" / "runs").glob("*.json"))), "runs done before")

In [ ]:
import torch

assert torch.cuda.is_available(), "no GPU: set the accelerator in the notebook settings"
print(torch.cuda.get_device_name(), torch.__version__)

## Runs

`teacher_<subset>` fine-tunes the 400M teacher on the labeled subset with cross-entropy (CE), then saves its logits on all 67,349 train sentences. `hard_<subset>` trains the student with CE on the same labels. `kd_<subset>` trains the student on the labeled subset with (1 - 0.5) CE + 0.5 T^2 KL(teacher || student) at temperature T = 2 ([Hinton et al., 2015](https://arxiv.org/abs/1503.02531)). `kdu_<subset>` uses the same loss on all 67k train sentences: CE where a gold label exists, KL everywhere. `p10` is a random 10% of the train set, `kc10` is the 10% chosen by k-center greedy on embeddings of the pretrained teacher ([Sener & Savarese, 2018](https://arxiv.org/abs/1708.00489)).

In [ ]:
!python kd.py {RUNS} --max-hours {MAX_HOURS}

## Results

`plots.py` averages the seeds of each config (mean ± sample std), writes `results/summary.md` and `results/figures/label_curve.png`.

In [ ]:
!python plots.py

In [ ]:
from IPython.display import Image, display

display(Image(str(WORK / "results" / "figures" / "label_curve.png")))

In [ ]:
shutil.make_archive(str(WORK / "results"), "zip", root_dir=WORK, base_dir="results")